# 🚀 Notebook do Professor (Demo) — Aula 13: LangGraph — StateGraph, nodes, conditional edges e HITL

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 13/14 — Módulo 4 · Grafos de estado em código**  
**1h40min**  
**StateGraph · MemorySaver · HITL · draw_mermaid**  
**Andaime 60%**  

---

## Como usar este notebook

- Cada célula corresponde a um slide de código da aula (a ordem é a da apresentação).
- Rode ao vivo enquanto explica o slide correspondente.
- A última seção traz as soluções completas dos exercícios de fixação.

---

# 🔬 Código da aula — slide a slide

### Slide 07 — StateGraph — estrutura minima funcional

In [ ]:
!pip install langchain-ollama langchain-core -q

from langchain_ollama import ChatOllama
from google.colab import userdata
import os

# Definir a API key via variável de ambiente (Colab Secrets)
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
!pip install langgraph langchain-ollama -q

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")
llm = ChatOllama(model="gpt-oss:120b", temperature=0)

# 1. Definir o Estado com TypedDict
class Estado(TypedDict):
    # Annotated[list, add_messages] usa reducer que ACUMULA (nao substitui)
    mensagens: Annotated[list, add_messages]
    iteracoes: int   # campo simples — substituido a cada update
    rota: str         # campo simples — a decisao do classificador

# 2. Definir Nodes — funcoes que recebem state e retornam dict
def node_responder(state: Estado) -> dict:
    resposta = llm.invoke(state["mensagens"])
    return {
        "mensagens": [resposta],  # add_messages ACUMULA — nao substitui
        "iteracoes": state["iteracoes"] + 1,
    }

# 3. Construir o Grafo
builder = StateGraph(Estado)
builder.add_node("responder", node_responder)
builder.add_edge(START, "responder")
builder.add_edge("responder", END)

# 4. Compilar — valida e retorna Runnable
app = builder.compile()

# 5. Invocar como qualquer chain LCEL
resultado = app.invoke({"mensagens": [HumanMessage("Oi!")], "iteracoes": 0, "rota": ""})
print(resultado["mensagens"][-1].content)

### Slide 09 — add_conditional_edges() — o superpoder do LangGraph

In [ ]:
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel
from typing import TypedDict, Annotated, Literal
from langgraph.graph.message import add_messages

class Estado(TypedDict):
    mensagens: Annotated[list, add_messages]
    rota: str
    iteracoes: int

class Rota(BaseModel):
    destino: Literal["rag", "calculadora", "conversa"]

# Node: classifica a intencao e escreve "rota" no estado
def node_classificar(state: Estado) -> dict:
    ultima_msg = state["mensagens"][-1].content
    rota_obj   = chain_clf.invoke({"input": ultima_msg})
    return {"rota": rota_obj.destino}

# Funcao de roteamento — le o estado e retorna string
def decidir_rota(state: Estado) -> str:
    return state["rota"]  # deve retornar uma das chaves do mapa abaixo

builder = StateGraph(Estado)
builder.add_node("classificar", node_classificar)
builder.add_node("rag",         node_rag)
builder.add_node("calculadora", node_calc)
builder.add_node("conversa",    node_chat)
builder.add_edge(START, "classificar")

# add_conditional_edges(no_origem, fn_rota, {retorno_fn: nome_no_destino})
builder.add_conditional_edges(
    "classificar",
    decidir_rota,
    {"rag":"rag", "calculadora":"calculadora", "conversa":"conversa"},
)
for no in ["rag", "calculadora", "conversa"]:
    builder.add_edge(no, END)
app = builder.compile()

### Slide 10 — draw_mermaid() — o grafo se auto-documenta

In [ ]:
from IPython.display import Image, display

# Renderizar como imagem PNG
try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    # Fallback: imprimir codigo Mermaid (colar em mermaid.live)
    print(app.get_graph().draw_mermaid())

# Saida do draw_mermaid() para o Router desta aula:
# graph TD
#     __start__([start]) --> classificar
#     classificar --> rag
#     classificar --> calculadora
#     classificar --> conversa
#     rag --> __end__([end])
#     calculadora --> __end__([end])
#     conversa --> __end__([end])
# Cole em https://mermaid.live para visualizar

# Inspecionar nos executados com stream()
for evento in app.stream({"mensagens": [HumanMessage("Qual o prazo?")],
                           "rota":"", "iteracoes":0}):
    print(f"No executado: {list(evento.keys())}")
# → No executado: ['classificar']
# → No executado: ['rag']

### Slide 12 — MemorySaver — estado persistente por thread_id

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()

# Compilar COM checkpointer — habilita persistencia e HITL
app = builder.compile(checkpointer=checkpointer)

# thread_id identifica a "sessao" — cada usuario tem seu proprio thread
config_u1 = {"configurable": {"thread_id": "usuario_01"}}
config_u2 = {"configurable": {"thread_id": "usuario_02"}}

# Chamada 1 — usuario 01
app.invoke(
    {"mensagens": [HumanMessage("Qual o prazo de garantia?")],
     "rota":"", "iteracoes":0},
    config=config_u1,
)

# Chamada 2 — mesmo usuario 01 — historico preservado
r = app.invoke(
    {"mensagens": [HumanMessage("E a multa por rescisao?")]},
    config=config_u1,
)
print(len(r["mensagens"]))  # → 4 (2 Human + 2 AI acumulados)

# Inspecionar o estado salvo
snapshot = app.get_state(config_u1)
print(f"Iteracoes: {snapshot.values['iteracoes']}")
print(f"Proximo no: {snapshot.next}")  # [] se terminou, ['nome'] se pausado

### Slide 14 — HITL com interrupt_before — aprovacao antes de acao irreversivel

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# Compilar COM interrupt_before — pausa ANTES do no especificado
app_hitl = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["enviar_email"],  # pausa ANTES deste no
)

config = {"configurable": {"thread_id": "tarefa_01"}}

# FASE 1 — rodar ate a pausa
app_hitl.invoke(
    {"mensagens": [HumanMessage("Gere um rascunho de email")]},
    config=config,
)
# → nos anteriores executaram, PAUSOU antes de "enviar_email"
# → estado salvo no MemorySaver — thread "suspensa"

# FASE 2 — humano le o rascunho
snapshot = app_hitl.get_state(config)
print(f"Rascunho: {snapshot.values['mensagens'][-1].content}")
print(f"No suspenso: {snapshot.next}")  # → ('enviar_email',)

# FASE 3a — humano aprova → continuar (input=None)
aprovado = input("Aprovar envio? (s/n): ")
if aprovado.lower() == "s":
    app_hitl.invoke(None, config=config)  # None = retomar sem novo input
    print("Email enviado!")
# FASE 3b — humano edita → update_state() antes de retomar
else:
    nova_msg = HumanMessage(input("Editar rascunho: "))
    app_hitl.update_state(config, {"mensagens": [nova_msg]})
    app_hitl.invoke(None, config=config)

### Slide 21 — Python novo desta aula

In [ ]:
# 1. TypedDict — dicionario com tipos declarados
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

class Estado(TypedDict):
    mensagens: Annotated[list, add_messages]  # reducer: acumula
    contador: int    # sem reducer: substituido no update

# 2. app.get_state(config) — inspecionar estado salvo
snapshot = app.get_state({"configurable":{"thread_id":"abc"}})
snapshot.values    # → dict com estado atual
snapshot.next      # → () se terminou, ('nome_no',) se pausado

# 3. app.update_state(config, values) — editar estado antes de retomar
app.update_state(
    {"configurable":{"thread_id":"abc"}},
    {"mensagens":[HumanMessage("versao editada")]},
)

# 4. app.invoke(None, config) — retomar grafo pausado
app.invoke(None, config=config)
# None = continuar do ponto suspenso sem novo input

# 5. pesquisador.get_graph().draw_mermaid() — auto-documentacao
print(pesquisador.get_graph().draw_mermaid())
# retorna string Mermaid — colar em mermaid.live para visualizar
# ou usar draw_mermaid_png() para renderizar inline no Colab

### Slide 26 — Demo Prática ao Vivo — agente pesquisador com loop condicional

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_community.tools import DuckDuckGoSearchRun
from pydantic import BaseModel
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate

class EstadoPesquisa(TypedDict):
    mensagens: Annotated[list, add_messages]
    resultados_busca: str
    qualidade: float
    iteracoes: int

class Qualidade(BaseModel):
    score: float; suficiente: bool

def node_buscar(state) -> dict:
    bruto = DuckDuckGoSearchRun().run(state["mensagens"][0].content)
    return {"resultados_busca":bruto[:1500], "iteracoes":state["iteracoes"]+1}

def node_avaliar(state) -> dict:
    q = (ChatPromptTemplate.from_template(
        "Pergunta:{p}\nResultado:{r}\nScore suficiencia 0.0-1.0:"
    ) | llm.with_structured_output(Qualidade)).invoke(
        {"p":state["mensagens"][0].content, "r":state["resultados_busca"]}
    )
    return {"qualidade": q.score}

def node_responder(state) -> dict:
    resp = llm.invoke([HumanMessage(
        f"Pergunta:{state['mensagens'][0].content}\nFonte:{state['resultados_busca']}"
    )])
    return {"mensagens":[resp]}

def decidir_continuar(state) -> str:
    # Para se qualidade OK ou atingiu limite de 3 iteracoes
    if state["qualidade"] >= 0.7 or state["iteracoes"] >= 3:
        return "responder"
    return "buscar"  # loop de volta

builder = StateGraph(EstadoPesquisa)
builder.add_node("buscar",node_buscar); builder.add_node("avaliar",node_avaliar)
builder.add_node("responder",node_responder)
builder.add_edge(START,"buscar"); builder.add_edge("buscar","avaliar")
builder.add_conditional_edges("avaliar",decidir_continuar,
                              {"buscar":"buscar","responder":"responder"})
builder.add_edge("responder",END)
pesquisador = builder.compile(checkpointer=MemorySaver())

---

## 🏋️ Exercícios Resolvidos — versão professor (executar no Colab)

As quatro soluções prontas dos exercícios de fixação do notebook do aluno — rode em sala, uma a uma.


### Exercício 1 — Monte o mini-grafo do Router

**O que a solução demonstra:** o andaime do Exercício 1 do aluno já preenchido — campo lido em `decidir_rota_ex`, nó faltante em `add_node`, aresta inicial e mapa completo de `add_conditional_edges` — com o grafo compilado e os 3 inputs terminando no nó correto.

**Pontos a destacar em sala:**
- Rodar a versão com lacunas ao lado (notebook do aluno) e esta: faltavam 4 valores — o campo do estado, o nome do nó, a aresta inicial e a entrada do mapa.
- Conferir no `draw_mermaid()` (célula do lab) que `classificar` tem 3 saídas: o mapa do `add_conditional_edges` é literalmente o 3º argumento.


In [ ]:
# Solução — mini-grafo do Router montado com LangGraph (Exercício 1)
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate

class EstadoEx(TypedDict):
    mensagens: Annotated[list, add_messages]
    rota: str

class RotaEx(BaseModel):
    destino: Literal["rag", "calculadora", "conversa"]

clf_ex = (ChatPromptTemplate.from_messages([
    ("system", """Você é um roteador de intenções. Classifique o input em exatamente um destino:
- rag: perguntas sobre documentos do domínio
- calculadora: cálculos numéricos
- conversa: saudações e qualquer outra coisa
Retorne apenas o JSON com o campo 'destino'."""),
    ("human", "{input}"),
]) | llm.with_structured_output(RotaEx))

def node_classificar(state: EstadoEx) -> dict:
    rota = clf_ex.invoke({"input": state["mensagens"][-1].content})
    return {"rota": rota.destino}

def node_rag(state: EstadoEx) -> dict:
    return {"mensagens": ["[RAG] resposta com base nos documentos do domínio"]}

def node_calc(state: EstadoEx) -> dict:
    return {"mensagens": ["[CALC] resultado da expressão"]}

def node_conversa(state: EstadoEx) -> dict:
    return {"mensagens": ["[CHAT] resposta amigável"]}

def decidir_rota_ex(state: EstadoEx) -> str:
    return state["rota"]

builder_ex = StateGraph(EstadoEx)
builder_ex.add_node("classificar", node_classificar)
builder_ex.add_node("rag", node_rag)
builder_ex.add_node("calculadora", node_calc)
builder_ex.add_node("conversa", node_conversa)
builder_ex.add_edge(START, "classificar")
builder_ex.add_conditional_edges("classificar", decidir_rota_ex,
    {"rag": "rag", "calculadora": "calculadora", "conversa": "conversa"})
for no in ["rag", "calculadora", "conversa"]:
    builder_ex.add_edge(no, END)

app_ex = builder_ex.compile()
for pergunta in ["Qual o prazo de garantia?", "Quanto é 12 × 0.9?", "Oi, tudo bem?"]:
    r = app_ex.invoke({"mensagens": [HumanMessage(pergunta)], "rota": ""})
    print(f"{pergunta[:28]:28s} → nó: {r['rota']:12s} | {r['mensagens'][-1].content[:40]}")


### Exercício 2 — Human-in-the-Loop: as 3 fases

**O que a solução demonstra:** o andaime do Exercício 2 preenchido — `interrupt_before=["revisar"]`, `get_state(config)` confirmando `('revisar',)` e a retomada com `invoke(None, config)` — mais o caminho de EDIÇÃO com `update_state` antes de retomar.

**Pontos a destacar em sala:**
- FASE 1: o grafo roda até `rascunhar` e PAUSA antes de `revisar`, estado salvo no `MemorySaver`; FASE 2: `snap.next == ('revisar',)` confirma a pausa; FASE 3: `invoke(None, config)` retoma do ponto exato — `Depois da retomada: ()`.
- O bloco final mostra a EDIÇÃO: `update_state(config, {...})` troca o estado salvo e só então `invoke(None, config)` — o padrão de aprovação/edição do slide de HITL.
- Reforço: sem checkpointer não há estado suspenso — o `interrupt_before` sozinho não pausa nada.


In [ ]:
# Solução — as 3 fases do HITL no mini-grafo (Exercício 2)
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
from langchain_core.messages import HumanMessage

class EstadoHitl(TypedDict):
    mensagens: Annotated[list, add_messages]
    rascunho: str

def node_rascunhar(state: EstadoHitl) -> dict:
    resp = llm.invoke(state["mensagens"])
    return {"mensagens": [resp], "rascunho": resp.content}

def node_revisar(state: EstadoHitl) -> dict:
    revisao = llm.invoke([HumanMessage(
        f"Revise e melhore o texto abaixo, mantendo o tamanho:\n{state['rascunho']}")])
    return {"mensagens": [revisao], "rascunho": revisao.content}

builder_hitl = StateGraph(EstadoHitl)
builder_hitl.add_node("rascunhar", node_rascunhar)
builder_hitl.add_node("revisar", node_revisar)
builder_hitl.add_edge(START, "rascunhar")
builder_hitl.add_edge("rascunhar", "revisar")
builder_hitl.add_edge("revisar", END)

app_hitl = builder_hitl.compile(checkpointer=MemorySaver(), interrupt_before=["revisar"])
config_hitl = {"configurable": {"thread_id": "fixacao13"}}

# FASE 1 — roda até a pausa (rascunhar executa; revisar fica suspenso)
app_hitl.invoke({"mensagens": [HumanMessage("Explique o que é um Router Chain em 2 linhas")],
                 "rascunho": ""}, config=config_hitl)

# FASE 2 — humano inspeciona o estado suspenso
snap = app_hitl.get_state(config_hitl)
print("Suspenso antes de:", snap.next)                # ('revisar',)
print("Rascunho:", snap.values["rascunho"])

# FASE 3 — humano aprova → retomar sem novo input
app_hitl.invoke(None, config=config_hitl)
print("Depois da retomada:", app_hitl.get_state(config_hitl).next)   # ()

# Caminho de EDIÇÃO — update_state antes de retomar (nova thread)
config_ed = {"configurable": {"thread_id": "fixacao13-edit"}}
app_hitl.invoke({"mensagens": [HumanMessage("Escreva um título de slide")],
                 "rascunho": ""}, config=config_ed)
app_hitl.update_state(config_ed, {"mensagens": [HumanMessage("versão editada pelo humano")]})
app_hitl.invoke(None, config=config_ed)
print("Depois da edição:", app_hitl.get_state(config_ed).next)


### Exercício 3 — Threshold e MAX do loop condicional

**O que a solução demonstra:** o andaime do Exercício 3 preenchido — `THRESHOLD=0.7` e `MAX_ITER=3` com os dois retornos de `decidir_continuar` — e o mesmo grafo recompilado em 0.5 / 0.7 / 0.9 sobre as 2 perguntas, mostrando iterações e score final.

**Pontos a destacar em sala:**
- Padrão esperado: `0.5` encerra na 1ª iteração (rápido e raso); `0.7` exige 1–2 iterações; `0.9` quase sempre esgota o `MAX 3` — um resumo de busca raramente recebe score tão alto.
- O threshold é o dial de latência × qualidade (cada iteração custa 1 busca + 1 avaliação); o `MAX` é o guard-rail que impede loop infinito.
- Provocar a turma: com qual configuração o grupo deixaria em produção? `0.6–0.7` com `MAX 3` costuma ser o equilíbrio.


In [ ]:
# Solução — loop condicional com THRESHOLD/MAX preenchidos + sweep de thresholds (Exercício 3)
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
from pydantic import BaseModel
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import DuckDuckGoSearchRun

class EstadoLoop(TypedDict):
    mensagens: Annotated[list, add_messages]
    resultados_busca: str
    qualidade: float
    iteracoes: int

class Qualidade(BaseModel):
    score: float; suficiente: bool

# Resposta das lacunas 1 e 2 — em produção, 0.6–0.7 com MAX 3 costuma ser o equilíbrio
THRESHOLD = 0.7
MAX_ITER = 3

def node_buscar_l(state: EstadoLoop) -> dict:
    bruto = DuckDuckGoSearchRun().run(state["mensagens"][0].content)
    return {"resultados_busca": bruto[:1500], "iteracoes": state["iteracoes"] + 1}

def node_avaliar_l(state: EstadoLoop) -> dict:
    q = (ChatPromptTemplate.from_template(
        "Pergunta: {p}\nResultado: {r}\nScore de suficiência 0.0-1.0:"
    ) | llm.with_structured_output(Qualidade)).invoke(
        {"p": state["mensagens"][0].content, "r": state["resultados_busca"]})
    return {"qualidade": q.score}

def node_responder_l(state: EstadoLoop) -> dict:
    resp = llm.invoke([HumanMessage(
        f"Pergunta: {state['mensagens'][0].content}\nFonte: {state['resultados_busca']}")])
    return {"mensagens": [resp]}

# Resposta das lacunas 3 e 4 — os dois retornos da função de roteamento
def decidir_continuar_l(state: EstadoLoop) -> str:
    if state["qualidade"] >= THRESHOLD or state["iteracoes"] >= MAX_ITER:
        return "responder"   # qualidade suficiente → responder
    return "buscar"          # senão → volta ao loop

def montar_pesquisador():
    builder_loop = StateGraph(EstadoLoop)
    builder_loop.add_node("buscar", node_buscar_l)
    builder_loop.add_node("avaliar", node_avaliar_l)
    builder_loop.add_node("responder", node_responder_l)
    builder_loop.add_edge(START, "buscar")
    builder_loop.add_edge("buscar", "avaliar")
    builder_loop.add_conditional_edges("avaliar", decidir_continuar_l,
                                       {"buscar": "buscar", "responder": "responder"})
    builder_loop.add_edge("responder", END)
    return builder_loop.compile(checkpointer=MemorySaver())

PERGUNTAS = ["O que é grounding em sistemas RAG?", "Como funciona o checkpointing do LangGraph?"]
for threshold in [0.5, 0.7, 0.9]:
    THRESHOLD = threshold
    pesquisador_l = montar_pesquisador()
    for pergunta in PERGUNTAS:
        config_l = {"configurable": {"thread_id": f"fix13-{threshold}-{pergunta[:12]}"}}
        pesquisador_l.invoke({"mensagens": [HumanMessage(pergunta)],
                              "resultados_busca": "", "qualidade": 0.0, "iteracoes": 0}, config=config_l)
        v = pesquisador_l.get_state(config_l).values
        esgotou = " · esgotou MAX" if v["iteracoes"] >= MAX_ITER else ""
        print(f"THRESHOLD={threshold} | iteracoes={v['iteracoes']} | "
              f"score={v['qualidade']:.2f}{esgotou} | {pergunta[:40]}")


### Exercício 4 — Fonte do domínio no node buscar

**O que a solução demonstra:** o andaime do Exercício 4 preenchido — `node_buscar` trocando a busca web pelo retriever Chroma do CKP02, grafo remontado com a aresta condicional de volta e compilado com checkpointer, mais a variante HITL com `interrupt_before=["responder"]`.

**Pontos a destacar em sala:**
- Só uma linha muda no `node_buscar` — `run(query)` vira `retriever.invoke(query)`; `avaliar` e `decidir_continuar` são agnósticos à fonte, e é isso que torna o `StateGraph` reutilizável entre domínios.
- No diagrama, conferir o ciclo: `buscar → avaliar`, a aresta de volta `avaliar → buscar` e o `avaliar → responder`.
- Para o HITL: pausar antes de ações com custo externo ou irreversível — no agente de documentos, tipicamente antes da resposta final entregue ao cliente.


In [ ]:
# Solução — o pesquisador com a fonte do CKP02 + variante HITL (Exercício 4)
!pip install langchain-chroma -q
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from IPython.display import Image, display

embeddings_ex = OllamaEmbeddings(model="nomic-embed-text")
retriever_ex = Chroma(persist_directory="/content/ckp02",
                      embedding_function=embeddings_ex).as_retriever(search_kwargs={"k": 3})

# Resposta da lacuna 1 — query do estado + join do conteúdo
def node_buscar_ex(state: EstadoLoop) -> dict:
    docs = retriever_ex.invoke(state["mensagens"][0].content)   # query = pergunta do estado
    bruto = "\n".join(d.page_content for d in docs)             # conteúdo de cada doc
    return {"resultados_busca": bruto[:1500],
            "iteracoes": state["iteracoes"] + 1}

# Reaproveita os nodes prontos da solução do Exercício 3
builder_ex4 = StateGraph(EstadoLoop)
builder_ex4.add_node("buscar", node_buscar_ex)
builder_ex4.add_node("avaliar", node_avaliar_l)
builder_ex4.add_node("responder", node_responder_l)
builder_ex4.add_edge(START, "buscar")
builder_ex4.add_edge("buscar", "avaliar")
builder_ex4.add_conditional_edges("avaliar", decidir_continuar_l,
                                  {"buscar": "buscar", "responder": "responder"})
builder_ex4.add_edge("responder", END)
pesquisador_ex4 = builder_ex4.compile(checkpointer=MemorySaver())

try:
    display(Image(pesquisador_ex4.get_graph().draw_mermaid_png()))
except Exception:
    print(pesquisador_ex4.get_graph().draw_mermaid())

config_ex4 = {"configurable": {"thread_id": "fix13-ckp02"}}
pesquisador_ex4.invoke({"mensagens": [HumanMessage("O que é grounding em RAG?")],
                        "resultados_busca": "", "qualidade": 0.0, "iteracoes": 0}, config=config_ex4)
print("Iteracoes:", pesquisador_ex4.get_state(config_ex4).values["iteracoes"])

# Variante HITL — pausar para um humano aprovar a resposta final
app_hitl_ex = builder_ex4.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["responder"],
)
config_h = {"configurable": {"thread_id": "pesquisador-hitl"}}
app_hitl_ex.invoke({"mensagens": [HumanMessage("O que é grounding em RAG?")],
                    "resultados_busca": "", "qualidade": 0.0, "iteracoes": 0}, config=config_h)
print("Suspenso antes de:", app_hitl_ex.get_state(config_h).next)   # ('responder',)


## 📚 Referências da aula

- Docs LangGraph — Guia completo: StateGraph, checkpointing, HITL. langchain-ai.github.io/langgraph/tutorials/introduction
- Docs LangGraph HITL — interrupt_before, update_state, invoke(None). langchain-ai.github.io/langgraph/concepts/human_in_the_loop
- Blog Anthropic Engineering — "Building Effective Agents" (2025). Secao sobre checkpointing e revisao humana. anthropic.com/engineering/building-effective-agents
- Tool Mermaid Live Editor — Para visualizar o output de draw_mermaid() sem instalar playwright. mermaid.live
- Livro Russell, S.; Norvig, P. — Inteligencia Artificial. 3ª ed. Pearson, 2016. Cap. 3 — Resolucao de problemas como busca: base conceitual dos grafos de estado no LangGraph.
- Livro Bornet, P.; Wirtz, J. et al. — Agentic Artificial Intelligence. World Scientific, 2025. A analogia do "funcionário recém-contratado" e HITL como fase de confiança, não trava permanente — a fundamentação por trás do interrupt_before desta aula.
- Livro Gullí, A. — Agentic Design Patterns. O'Reilly, 2025. Cap. 4 — Reflection: o modelo Producer-Critic que justifica separar os nós buscar e avaliar no grafo pesquisador desta aula.

---

**Proxima Aula — Aula 14 (ultima)** — Spec-Driven Development e Encerramento
  
Retrospectiva do semestre · Spec-Driven Development · LangSmith · Proximos passos na carreira · Encerramento.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*